# sdimg visual test notebook

This notebook uses `asset/sample_image.png` and `asset/sample_mask.png` to visually inspect the main `sdimg` functions.
Run cells from top to bottom.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

import sdimg

In [ ]:
def bgr_to_rgb(image: np.ndarray) -> np.ndarray:
    if image.ndim == 2:
        return image
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def show_images(items: list[tuple[str, np.ndarray]], cols: int = 3, figsize: tuple[int, int] = (15, 10)) -> None:
    rows = (len(items) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.atleast_1d(axes).ravel()

    for ax in axes:
        ax.axis("off")

    for ax, (title, image) in zip(axes, items):
        ax.set_title(title)
        if image.ndim == 2:
            ax.imshow(image, cmap="gray")
        else:
            ax.imshow(bgr_to_rgb(image))

    plt.tight_layout()
    plt.show()


def show_masks(items: list[tuple[str, np.ndarray]], cols: int = 3, figsize: tuple[int, int] = (15, 10)) -> None:
    rows = (len(items) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.atleast_1d(axes).ravel()

    for ax in axes:
        ax.axis("off")

    for ax, (title, mask) in zip(axes, items):
        ax.set_title(title)
        ax.imshow(mask, cmap="gray", vmin=0, vmax=1)

    plt.tight_layout()
    plt.show()

In [ ]:
image = cv2.imread("asset/sample_image.png", cv2.IMREAD_COLOR)
mask_image = cv2.imread("asset/sample_mask.png", cv2.IMREAD_GRAYSCALE)
if image is None:
    raise FileNotFoundError("asset/sample_image.png not found")
if mask_image is None:
    raise FileNotFoundError("asset/sample_mask.png not found")

gray = sdimg.to_gray(image)
base_mask = sdimg.to_mask(mask_image)

print("image:", image.shape, image.dtype)
print("gray:", gray.shape, gray.dtype)
print("mask:", base_mask.shape, base_mask.dtype, np.unique(base_mask))

## Base data

In [ ]:
show_images([
    ("original", image),
    ("gray", gray),
])

show_masks([
    ("base mask", base_mask),
])

## Core

In [ ]:
rgb_from_gray = sdimg.to_rgb(gray)
normalized_mask = sdimg.to_mask(base_mask * 255)

print("is_image(image):", sdimg.is_image(image))
print("is_mask(base_mask):", sdimg.is_mask(base_mask))
print("rgb_from_gray:", rgb_from_gray.shape, rgb_from_gray.dtype)
print("normalized_mask unique:", np.unique(normalized_mask))

show_images([
    ("gray", gray),
    ("to_rgb(gray)", rgb_from_gray),
])

## Image

In [ ]:
image_results = [
    ("standard_norm", sdimg.standard_norm(image)),
    ("hist_norm", sdimg.hist_norm(image)),
    ("clahe_norm", sdimg.clahe_norm(image)),
    ("brightness_contrast", sdimg.adjust_brightness_contrast(image, brightness=0.1, contrast=0.2)),
    ("gaussian_blur", sdimg.gaussian_blur(image, (5, 5), 1.2)),
    ("median_blur", sdimg.median_blur(image, 5)),
    ("denoise", sdimg.denoise(image)),
    ("sharpen", sdimg.sharpen(image, alpha=1.0)),
]

show_images([("original", image)] + image_results, cols=3, figsize=(15, 16))

## Mask

In [ ]:
mask_open = sdimg.morphology(base_mask, "open", (5, 5), 1)
mask_close = sdimg.morphology(base_mask, "close", (5, 5), 1)
mask_holes = sdimg.fill_holes(base_mask)
mask_largest = sdimg.keep_largest_component(base_mask)
mask_convex = sdimg.convex_hull(base_mask)
mask_concave = sdimg.concave_hull(base_mask)
mask_edge = sdimg.extract_edge(base_mask)
mask_distance = sdimg.distance_transform(base_mask)

print("coords shape:", sdimg.get_coords(base_mask).shape)
print("bbox:", sdimg.get_bbox(base_mask))
print("area:", sdimg.get_area(base_mask))
print("centroid:", sdimg.get_centroid(base_mask))

show_masks([
    ("base", base_mask),
    ("open", mask_open),
    ("close", mask_close),
    ("fill_holes", mask_holes),
    ("largest", mask_largest),
    ("convex_hull", mask_convex),
    ("concave_hull", mask_concave),
    ("edge", mask_edge),
], cols=4, figsize=(16, 12))

plt.figure(figsize=(6, 6))
plt.title("distance_transform")
plt.imshow(mask_distance, cmap="magma")
plt.axis("off")
plt.colorbar()
plt.show()

## Spatial

In [ ]:
bbox = sdimg.get_bbox(base_mask)
if bbox is None:
    raise ValueError("base_mask is empty")

resized = sdimg.resize(image, height=512)
rotated = sdimg.rotate(image, 90)
flipped = sdimg.flip(image, "horizontal")
padded, pad_meta = sdimg.pad(image, 40, return_meta=True)
unpadded = sdimg.unpad(padded, pad_meta)
cropped = sdimg.crop(image, bbox)
patches, split_meta = sdimg.split(image, n=3, overlap=0.25, return_meta=True)
merged = sdimg.merge(patches, split_meta)

show_images([
    ("original", image),
    ("resize", resized),
    ("rotate 90", rotated),
    ("flip horizontal", flipped),
    ("pad", padded),
    ("unpad", unpadded),
    ("crop", cropped),
    ("merge(split)", merged),
], cols=3, figsize=(15, 16))

show_images([(f"patch {idx}", patch) for idx, patch in enumerate(patches[:6])], cols=3, figsize=(12, 8))

## Fusion

In [ ]:
otsu_mask = sdimg.otsu_threshold(image)
initial_mask = sdimg.keep_largest_component(otsu_mask)
grabcut_mask = sdimg.grabcut_refine(image, initial_mask)
guided_mask = sdimg.guided_filter_refine(image, initial_mask)

show_masks([
    ("otsu", otsu_mask),
    ("initial", initial_mask),
    ("grabcut", grabcut_mask),
    ("guided_filter", guided_mask),
], cols=3, figsize=(15, 10))

In [ ]:
sync_resized_image, sync_resized_mask = sdimg.sync_resize(image, initial_mask, height=512)
sync_rotated_image, sync_rotated_mask = sdimg.sync_rotate(image, initial_mask, 90)
sync_flipped_image, sync_flipped_mask = sdimg.sync_flip(image, initial_mask, "horizontal")
sync_padded_image, sync_padded_mask, sync_pad_meta = sdimg.sync_pad(image, initial_mask, 40, return_meta=True)
sync_unpadded_image, sync_unpadded_mask = sdimg.sync_unpad(sync_padded_image, sync_padded_mask, sync_pad_meta)
sync_cropped_image, sync_cropped_mask = sdimg.sync_crop(image, initial_mask, bbox)
sync_image_patches, sync_mask_patches, sync_meta = sdimg.sync_split(image, initial_mask, n=3, overlap=0.25, return_meta=True)
sync_merged_image, sync_merged_mask = sdimg.sync_merge(sync_image_patches, sync_mask_patches, sync_meta)

show_images([
    ("sync resize image", sync_resized_image),
    ("sync rotate image", sync_rotated_image),
    ("sync flip image", sync_flipped_image),
    ("sync pad image", sync_padded_image),
    ("sync unpad image", sync_unpadded_image),
    ("sync crop image", sync_cropped_image),
    ("sync merge image", sync_merged_image),
], cols=3, figsize=(15, 14))

show_masks([
    ("sync resize mask", sync_resized_mask),
    ("sync rotate mask", sync_rotated_mask),
    ("sync flip mask", sync_flipped_mask),
    ("sync pad mask", sync_padded_mask),
    ("sync unpad mask", sync_unpadded_mask),
    ("sync crop mask", sync_cropped_mask),
    ("sync merge mask", sync_merged_mask),
], cols=3, figsize=(15, 14))